# 01 - Dataset Loading

**Phase 1, Step 1.1 - Dataset Loading**

**Source: official COCO 2017 dataset (`cocodataset.org`)**, not the Hugging Face streaming mirror. The `detection-datasets/coco` HF mirror hit a server-side outage (its storage backend was returning `403 Forbidden` / malformed HTML instead of data), so this notebook uses the same source named in the project brief instead: `https://cocodataset.org/` via the official `pycocotools` API.

This approach downloads only the ~241MB annotation file (not the full ~18GB image set), uses it to look up exactly which images contain our 25 target classes, and downloads only those specific images (a few thousand, not all 118K) directly from `images.cocodataset.org` in parallel.

**Bonus:** the official annotation JSON includes real category names directly - no more guessing at category ID mappings. Bounding boxes are converted from COCO's official `[x, y, width, height]` format to `[x_min, y_min, x_max, y_max]` at collection time here, so every downstream notebook (02, 03, 10) keeps working completely unchanged - they already expect the corner format.

Everything is saved to **Google Drive** under `SmartVisionAI/raw_data/` so that every later notebook (EDA, preprocessing, YOLO dataset prep) can reuse this exact same raw collection - even in a brand new Colab session.

### 1. Install dependencies and mount Drive

In [1]:
!pip install -q datasets pyyaml tqdm scikit-learn matplotlib pandas opencv-python-headless pycocotools

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### 2. Project configuration

In [3]:
import os

# ---------------------------------------------------------------------------
# Project configuration - shared across every SmartVision AI notebook.
# All notebooks read/write under this same Google Drive folder so that
# work done in one notebook (e.g. dataset collection) is available to the
# next one (e.g. training), even across separate Colab sessions.
# ---------------------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/SmartVisionAI"

RAW_DATA_DIR = os.path.join(BASE_DIR, "raw_data")
RAW_IMAGES_DIR = os.path.join(RAW_DATA_DIR, "images")
RAW_ANNOTATIONS_PATH = os.path.join(RAW_DATA_DIR, "annotations.json")

CLASSIFICATION_DIR = os.path.join(BASE_DIR, "classification")
CLASSIFICATION_TRAIN_DIR = os.path.join(CLASSIFICATION_DIR, "train")
CLASSIFICATION_VAL_DIR = os.path.join(CLASSIFICATION_DIR, "val")
CLASSIFICATION_TEST_DIR = os.path.join(CLASSIFICATION_DIR, "test")

DETECTION_DIR = os.path.join(BASE_DIR, "detection")
DETECTION_IMAGES_DIR = os.path.join(DETECTION_DIR, "images")
DETECTION_LABELS_DIR = os.path.join(DETECTION_DIR, "labels")
DETECTION_YAML_PATH = os.path.join(DETECTION_DIR, "data.yaml")

MODELS_DIR = os.path.join(BASE_DIR, "models")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

for d in [BASE_DIR, RAW_DATA_DIR, RAW_IMAGES_DIR, CLASSIFICATION_DIR, DETECTION_DIR, MODELS_DIR, OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# The 25 selected COCO classes (must match COCO category names exactly)
SELECTED_CLASSES = [
    # Vehicles (6)
    "car", "truck", "bus", "motorcycle", "bicycle", "airplane",
    # Person (1)
    "person",
    # Outdoor (3)
    "traffic light", "stop sign", "bench",
    # Animals (6)
    "dog", "cat", "horse", "bird", "cow", "elephant",
    # Kitchen & food (5)
    "bottle", "cup", "bowl", "pizza", "cake",
    # Furniture & indoor (4)
    "chair", "couch", "bed", "potted plant",
]
assert len(SELECTED_CLASSES) == 25

CLASS_TO_IDX = {name: i for i, name in enumerate(SELECTED_CLASSES)}
IDX_TO_CLASS = {i: name for i, name in enumerate(SELECTED_CLASSES)}

def safe_name(class_name):
    return class_name.replace(" ", "_")

IMAGES_PER_CLASS = 350        # -> 8,750 images total (up from 100/class to fight overfitting)
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

CLS_IMG_SIZE = 224            # Classification input resolution (single-resolution throughout)
FINE_TUNE_IMG_SIZE = 384      # Unused by classifier training (reverted to single-resolution); kept for compatibility
YOLO_IMG_SIZE = 640
BATCH_SIZE = 32                # Stage 1 batch size
BATCH_SIZE_STAGE2 = 16         # Smaller batch at 384x384 to fit GPU memory (~2.9x pixels/image)

HF_DATASET_NAME = "detection-datasets/coco"

print("BASE_DIR:", BASE_DIR)
print("Classes:", len(SELECTED_CLASSES))


BASE_DIR: /content/drive/MyDrive/SmartVisionAI
Classes: 25


### 3. Download the official COCO 2017 annotations

Only `instances_train2017.json` is extracted from the annotations zip - the captions/keypoints files inside the same archive aren't needed and are discarded to save disk space.

In [4]:
import os
import zipfile
import urllib.request

COCO_ANNOTATIONS_ZIP_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
COCO_LOCAL_DIR = "/content/coco_annotations"
os.makedirs(COCO_LOCAL_DIR, exist_ok=True)

INSTANCES_JSON_PATH = os.path.join(COCO_LOCAL_DIR, "annotations", "instances_train2017.json")

if not os.path.exists(INSTANCES_JSON_PATH):
    zip_path = os.path.join(COCO_LOCAL_DIR, "annotations_trainval2017.zip")
    print("Downloading official COCO 2017 annotations (~241MB)...")
    urllib.request.urlretrieve(COCO_ANNOTATIONS_ZIP_URL, zip_path)
    print("Extracting instances_train2017.json...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extract("annotations/instances_train2017.json", COCO_LOCAL_DIR)
    os.remove(zip_path)  # free disk space - we only needed one file from the zip
    print("Done.")
else:
    print("Annotations already downloaded.")


Extracting instances_train2017.json...
Done.


### 4. Load annotations with pycocotools and resolve our 25 classes

In [5]:
from pycocotools.coco import COCO

coco = COCO(INSTANCES_JSON_PATH)

all_cats = coco.loadCats(coco.getCatIds())
cat_id_to_name = {c["id"]: c["name"] for c in all_cats}
name_to_cat_id = {v: k for k, v in cat_id_to_name.items()}

missing = [c for c in SELECTED_CLASSES if c not in name_to_cat_id]
assert not missing, f"Classes not found in official COCO categories: {missing}"
print(f"Resolved all {len(SELECTED_CLASSES)} classes to official COCO category IDs.")


loading annotations into memory...
Done (t=16.84s)
creating index...
index created!
Resolved all 25 classes to official COCO category IDs.


### 5. Determine exactly which images to download (metadata only, no downloads yet)

`pycocotools` gives us direct index lookups (which images contain which categories), so - unlike streaming - we know in advance exactly which images we need, with no wasted scanning.

In [6]:
import random
random.seed(42)

collected_counts = {c: 0 for c in SELECTED_CLASSES}
images_to_download = {}  # image_id -> {"file", "width", "height", "boxes", "url"}

for cname in SELECTED_CLASSES:
    cid = name_to_cat_id[cname]
    img_ids = coco.getImgIds(catIds=[cid])
    random.shuffle(img_ids)

    for img_id in img_ids:
        if collected_counts[cname] >= IMAGES_PER_CLASS:
            break
        if img_id not in images_to_download:
            img_info = coco.loadImgs(img_id)[0]
            ann_ids = coco.getAnnIds(imgIds=img_id)
            anns = coco.loadAnns(ann_ids)

            image_boxes = []
            for a in anns:
                box_cname = cat_id_to_name.get(a["category_id"])
                if box_cname in SELECTED_CLASSES:
                    # Official COCO format is [x, y, width, height] - convert to
                    # [x_min, y_min, x_max, y_max] here so every downstream notebook
                    # (02, 03, 10) keeps working unchanged, since they already expect
                    # the corner format.
                    x, y, w, h = a["bbox"]
                    image_boxes.append({"class": box_cname, "bbox": [x, y, x + w, y + h]})

            if not image_boxes:
                continue

            images_to_download[img_id] = {
                "width": img_info["width"],
                "height": img_info["height"],
                "boxes": image_boxes,
                "url": img_info["coco_url"],
            }

        # Count this image toward every under-quota class it contains
        for box in images_to_download[img_id]["boxes"]:
            if collected_counts[box["class"]] < IMAGES_PER_CLASS:
                collected_counts[box["class"]] += 1

print(f"Need to download {len(images_to_download)} unique images.")
for c, n in collected_counts.items():
    flag = "" if n >= IMAGES_PER_CLASS else "  <-- short"
    print(f"  {c:<15} {n}{flag}")


Need to download 3087 unique images.
  car             350
  truck           350
  bus             350
  motorcycle      350
  bicycle         350
  airplane        350
  person          350
  traffic light   350
  stop sign       350
  bench           350
  dog             350
  cat             350
  horse           350
  bird            350
  cow             350
  elephant        350
  bottle          350
  cup             350
  bowl            350
  pizza           350
  cake            350
  chair           350
  couch           350
  bed             350
  potted plant    350


### 6. Download the needed images in parallel

Only the images identified above are downloaded (a few thousand, not the full 118K-image train2017 set), using a thread pool since this is a network-bound task.

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import json

def download_one(item):
    img_id, meta, fname = item
    local_path = os.path.join(RAW_IMAGES_DIR, fname)
    try:
        urllib.request.urlretrieve(meta["url"], local_path)
        return (img_id, fname, meta, True, None)
    except Exception as e:
        return (img_id, fname, meta, False, str(e))

download_jobs = [
    (img_id, meta, f"img_{i:05d}.jpg")
    for i, (img_id, meta) in enumerate(images_to_download.items())
]

annotations = []
failed = 0

with ThreadPoolExecutor(max_workers=32) as executor:
    futures = [executor.submit(download_one, job) for job in download_jobs]
    for future in tqdm(as_completed(futures), total=len(futures)):
        img_id, fname, meta, ok, err = future.result()
        if ok:
            annotations.append({
                "file": fname,
                "width": meta["width"],
                "height": meta["height"],
                "boxes": meta["boxes"],
            })
        else:
            failed += 1

print(f"Downloaded {len(annotations)} images successfully ({failed} failed).")


100%|██████████| 3087/3087 [00:50<00:00, 61.40it/s]

Downloaded 3087 images successfully (0 failed).


### 7. Save annotations and a metadata summary

In [8]:
with open(RAW_ANNOTATIONS_PATH, "w") as f:
    json.dump(annotations, f)

# Recompute final per-class counts from what was actually downloaded successfully
final_counts = {c: 0 for c in SELECTED_CLASSES}
for ann in annotations:
    for box in ann["boxes"]:
        final_counts[box["class"]] += 1

metadata = {
    "classes": SELECTED_CLASSES,
    "images_per_class_target": IMAGES_PER_CLASS,
    "actual_counts": final_counts,
    "total_unique_images": len(annotations),
    "source": "official COCO 2017 (cocodataset.org) via pycocotools",
}
with open(os.path.join(RAW_DATA_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Final per-class counts (boxes actually downloaded):")
for c, n in final_counts.items():
    print(f"  {c:<15} {n}")

print(f"\nSaved annotations to {RAW_ANNOTATIONS_PATH}")
print(f"Saved images to {RAW_IMAGES_DIR}")


Final per-class counts (boxes actually downloaded):
  car             2278
  truck           659
  bus             418
  motorcycle      406
  bicycle         452
  airplane        350
  person          6786
  traffic light   467
  stop sign       358
  bench           411
  dog             430
  cat             422
  horse           354
  bird            375
  cow             355
  elephant        350
  bottle          766
  cup             825
  bowl            482
  pizza           349
  cake            347
  chair           1043
  couch           379
  bed             344
  potted plant    348

Saved annotations to /content/drive/MyDrive/SmartVisionAI/raw_data/annotations.json
Saved images to /content/drive/MyDrive/SmartVisionAI/raw_data/images


**Next notebook:** `02_EDA.ipynb` analyzes this raw collection before we crop anything for classification or build the YOLO detection dataset.